In [112]:
from databricks.sdk import WorkspaceClient
from datetime import datetime, timedelta
import requests
import xml.etree.ElementTree as ET
from isodate import parse_duration
import pandas as pd

In [36]:
w = WorkspaceClient()
api_key = w.dbutils.secrets.get(scope='demand-pipeline', key='entsoe-api-key')

In [37]:
url = "https://web-api.tp.entsoe.eu/api"

payload = {
    'securityToken': api_key,
    'documentType': 'A65',
    'processType': 'A16', # actual total load
    'outBiddingZone_Domain': '10YCH-SWISSGRIDZ', # Switzerland
    'periodStart': '202201010000',
    #'periodEnd': f"{(datetime.now() - timedelta(1)).strftime('%Y%m%d')}0000"
    'periodEnd': '202201020000'
}

In [38]:
response = requests.get(url=url, params=payload)
response.raise_for_status()

In [128]:
xml = response.text

In [92]:
root = ET.fromstring(xml)

In [106]:
ns = {"ns": f"{root[0].tag.removeprefix('{').removesuffix('}mRID')}"}

In [137]:
def parse_xml_load(xml_text):
    root = ET.fromstring(xml_text)
    ns = {"ns": f"{root[0].tag.removeprefix('{').removesuffix('}mRID')}"}
    rows = []

    for ts in root.findall("ns:TimeSeries", ns):
        for period in ts.findall("ns:Period", ns):
            start = pd.to_datetime(period.find("ns:timeInterval/ns:start", ns).text)
            resolution = parse_duration(period.find("ns:resolution", ns).text)

            for point in period.findall("ns:Point", ns):
                position = int(point.find("ns:position", ns).text)
                quantity = float(point.find("ns:quantity", ns).text)
                timestamp = start + (position - 1) * resolution
                
                rows.append({
                    'datetime': timestamp,
                    'load_mw': quantity
                             })
    
    return pd.DataFrame(rows)

In [138]:
df = parse_xml_load(xml_text = xml)